# Loss Functions

**IMSE 774 · Neural Networks · Week 5 · Sessions 9–10 · September 22 & 24, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/05-loss-functions.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

## Maximizing log-likelihood


`nll-underflow`


In [ ]:
rng = np.random.default_rng(774)
resid = rng.normal(0, 1, 2000)
probs = np.exp(-0.5 * resid ** 2) / np.sqrt(2 * np.pi)   # Pr(y_i | f[x_i, phi])

print(f"smallest single factor: {probs.min():.3e}   (nothing unusual)")
print(f"\n{'I':>6}{'product (eq. 5.1)':>22}{'sum of logs (eq. 5.3)':>24}")
for I in (100, 300, 500, 2000):
    print(f"{I:>6}{np.prod(probs[:I]):>22.3e}{np.log(probs[:I]).sum():>24.2f}")

## The least squares loss function


`fig-ls-nll`


In [ ]:
x_d = np.linspace(0, 2, 12)
y_d = 0.85 + 0.45 * x_d + rng.normal(0, 0.12, x_d.shape)

def sse(slope, intercept=0.85):                       # equation (5.11)
    return np.sum((y_d - (intercept + slope * x_d)) ** 2)

def nll_normal(slope, sigma2, intercept=0.85):        # equation (5.9)
    f = intercept + slope * x_d
    return -np.sum(-0.5 * np.log(2 * np.pi * sigma2) - (y_d - f) ** 2 / (2 * sigma2))

slopes = np.linspace(-0.2, 1.1, 400)
L_ls = np.array([sse(s) for s in slopes])
best = slopes[L_ls.argmin()]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(x_d, y_d, s=28, color=ACCENT, zorder=3)
axes[0].plot(x_d, 0.85 + best * x_d, color=NDSU_GREEN, lw=2)
axes[0].set_title("data and least-squares fit"); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[1].plot(slopes, L_ls / L_ls.max(), color=NDSU_GREEN, lw=2.4, label="least squares (scaled)")
for s2, ls in zip((0.01, 0.05, 0.25), ("-", "--", ":")):
    L = np.array([nll_normal(s, s2) for s in slopes])
    axes[1].plot(slopes, (L - L.min()) / (L - L.min()).max(), ls, color=ACCENT, lw=1.4,
                 label=rf"$-\log$ likelihood, $\sigma^2 = {s2}$")
axes[1].axvline(best, color="gray", ls=":", lw=1)
axes[1].set_title("both criteria, rescaled to $[0, 1]$"); axes[1].set_xlabel("slope"); axes[1].set_ylabel("loss")
axes[1].legend(fontsize=7.5)
plt.tight_layout()
plt.show()

print(f"argmin slope, least squares: {best:.4f}\n")
print(f"{'sigma^2':>9}{'argmin slope':>14}{'max |NLL - (SSE/2s^2 + I/2 log 2 pi s^2)|':>45}")
for s2 in (0.01, 0.05, 0.25):
    L = np.array([nll_normal(s, s2) for s in slopes])
    affine = L_ls / (2 * s2) + len(x_d) / 2 * np.log(2 * np.pi * s2)
    print(f"{s2:>9}{slopes[L.argmin()]:>14.4f}{np.abs(L - affine).max():>45.2e}")

## Heteroscedastic regression


`fig-heteroscedastic`


In [ ]:
def true_mean(x):  return 0.4 * np.sin(3.0 * x) - 0.25 * x
def true_sd(x):    return 0.03 + 0.22 * x

x_h = np.random.uniform(0, 2, 600)
y_h = true_mean(x_h) + np.random.normal(0, 1, x_h.shape) * true_sd(x_h)
X_h = torch.tensor(x_h, dtype=torch.float32)[:, None]
Y_h = torch.tensor(y_h, dtype=torch.float32)[:, None]
x_grid = torch.linspace(0, 2, 400)[:, None]

def shallow_net(D_o, D=40):
    return nn.Sequential(nn.Linear(1, D), nn.ReLU(), nn.Linear(D, D_o))

def homo_loss(out):                                  # equation (5.11)
    return ((Y_h - out) ** 2).mean()

def hetero_loss(out):                                # equation (5.15)
    mu, sigma2 = out[:, 0:1], out[:, 1:2] ** 2 + 1e-6   # the epsilon keeps log finite
    return (0.5 * torch.log(2 * np.pi * sigma2) + (Y_h - mu) ** 2 / (2 * sigma2)).mean()

def train(model, loss_fn, epochs=4000, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X_h))
        loss.backward()
        opt.step()
    return loss.item()

torch.manual_seed(774); homo   = shallow_net(1)
torch.manual_seed(774); hetero = shallow_net(2)
train(homo, homo_loss)
train(hetero, hetero_loss)

with torch.no_grad():
    mu_h = homo(X_h).squeeze().numpy()
    sd_h = np.sqrt(np.mean((y_h - mu_h) ** 2))          # equation (5.13): the MLE of a constant sigma
    out_t = hetero(X_h)
    mu_t, sd_t = out_t[:, 0].numpy(), np.abs(out_t[:, 1].numpy())
    g = x_grid.squeeze().numpy()
    mg_h = homo(x_grid).squeeze().numpy()
    og = hetero(x_grid)
    mg_t, sg_t = og[:, 0].numpy(), np.abs(og[:, 1].numpy())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, m, s, title in [(axes[0], mg_h, np.full_like(g, sd_h), "homoscedastic: least squares"),
                        (axes[1], mg_t, sg_t, "heteroscedastic: equation (5.15)")]:
    ax.scatter(x_h, y_h, s=5, color="gray", alpha=0.35)
    ax.fill_between(g, m - 2 * s, m + 2 * s, color=NDSU_GREEN, alpha=0.18, label=r"$\pm 2\sigma$")
    ax.plot(g, m, color=NDSU_GREEN, lw=2, label=r"predicted $\mu$")
    ax.plot(g, true_mean(g) + 2 * true_sd(g), "--", color=ACCENT, lw=1.1, label=r"true $\pm 2\sigma$")
    ax.plot(g, true_mean(g) - 2 * true_sd(g), "--", color=ACCENT, lw=1.1)
    ax.set_title(title); ax.set_xlabel("$x$"); ax.set_ylim(-1.6, 1.6)
axes[0].set_ylabel("$y$"); axes[0].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

def mean_nll(mu, sd):  return float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + (y_h - mu) ** 2 / (2 * sd ** 2)))
def coverage(mask, mu, sd):
    sd = np.full_like(mu, sd) if np.ndim(sd) == 0 else sd
    return float(np.mean(np.abs(y_h[mask] - mu[mask]) <= 2 * sd[mask]))

low, high, allpts = x_h < 0.7, x_h > 1.3, np.ones_like(x_h, dtype=bool)
print(f"{'model':<20}{'mean NLL':>10}{'in +/-2s, all':>15}{'x < 0.7':>10}{'x > 1.3':>10}")
print(f"{'homoscedastic':<20}{mean_nll(mu_h, sd_h):>10.3f}{coverage(allpts, mu_h, sd_h):>15.1%}{coverage(low, mu_h, sd_h):>10.1%}{coverage(high, mu_h, sd_h):>10.1%}")
print(f"{'heteroscedastic':<20}{mean_nll(mu_t, sd_t):>10.3f}{coverage(allpts, mu_t, sd_t):>15.1%}{coverage(low, mu_t, sd_t):>10.1%}{coverage(high, mu_t, sd_t):>10.1%}")
print(f"{'nominal for +/-2s':<20}{'':>10}{0.9545:>15.1%}{0.9545:>10.1%}{0.9545:>10.1%}")
print(f"\nhomoscedastic sigma: {sd_h:.3f} everywhere")
print(f"heteroscedastic sigma: {sg_t.min():.3f} at the left edge, rising to {sg_t.max():.3f} at the right")

## Example 2: Binary Classification


`fig-binary`


In [ ]:
def pwl_network(knots, values):
    """The shallow ReLU network whose output is the piecewise linear function
    through (knots, values). One hidden unit per interior interval."""
    knots, values = np.asarray(knots, float), np.asarray(values, float)
    slopes = np.diff(values) / np.diff(knots)
    theta = np.column_stack([-knots[:-1], np.ones(len(knots) - 1)])
    phi = np.concatenate([[slopes[0]], np.diff(slopes)])
    return theta, values[0], phi

def shallow(x, theta, phi0, phi):
    pre = theta[:, 0:1] + theta[:, 1:2] * x[None, :]
    return phi0 + phi @ np.maximum(pre, 0)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

KNOTS = [0.0, 0.45, 1.0, 1.6, 2.0]
theta_b, phi0_b, phi_b = pwl_network(KNOTS, [-4.0, 3.5, -0.5, 3.0, -3.0])

x_b = np.linspace(0, 2, 1000)
f_b = shallow(x_b, theta_b, phi0_b, phi_b)
lam = sigmoid(f_b)

x_i = np.sort(np.random.uniform(0, 2, 60))
lam_i = sigmoid(shallow(x_i, theta_b, phi0_b, phi_b))
y_i = (np.random.uniform(size=x_i.shape) < lam_i).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.2))
axes[0].plot(x_b, f_b, color=STEEL, lw=2.2); axes[0].axhline(0, color="k", lw=0.6)
axes[0].set_title(r"a) $\mathrm{f}[x, \phi]$"); axes[0].set_ylabel(r"$\mathrm{f}[x, \phi]$")
z = np.linspace(-5, 5, 400)
axes[1].plot(z, sigmoid(z), color=ACCENT, lw=2.2)
axes[1].set_title(r"b) $\mathrm{sig}[z]$"); axes[1].set_xlabel("$z$"); axes[1].set_ylabel(r"$\mathrm{sig}[z]$")
axes[2].plot(x_b, lam, color=NDSU_GREEN, lw=2.2, label=r"$\lambda = Pr(y=1\mid x)$")
axes[2].plot(x_b, 1 - lam, "--", color=NDSU_GREEN, lw=1.4, label=r"$1 - \lambda$")
axes[2].scatter(x_i, y_i, s=16, color=ACCENT, zorder=3, alpha=0.75, label="sampled labels")
axes[2].set_title(r"c) $\lambda = \mathrm{sig}[\mathrm{f}[x, \phi]]$"); axes[2].legend(fontsize=7.5, loc="center right")
for ax in (axes[0], axes[2]):
    ax.set_xlabel("$x$")
    for j in KNOTS[1:-1]:
        ax.axvline(j, color="gray", ls=":", lw=0.9)
plt.tight_layout()
plt.show()

loss = np.sum(-(1 - y_i) * np.log(1 - lam_i) - y_i * np.log(lam_i))    # equation (5.20)
coin = len(y_i) * np.log(2)                                            # always answer lambda = 0.5

print(f"{y_i.sum()} of {len(y_i)} sampled labels are positive\n")
print(f"{'model':<28}{'total loss':>12}{'per example':>13}")
print(f"{'the true lambda[x]':<28}{loss:>12.3f}{loss / len(y_i):>13.3f}")
print(f"{'a constant lambda = 0.5':<28}{coin:>12.3f}{coin / len(y_i):>13.3f}")
print(f"\naccuracy of thresholding lambda at 0.5: {np.mean((lam_i > 0.5) == y_i):.1%}")

`fig-bce-curves`


In [ ]:
lam_axis = np.linspace(1e-4, 1 - 1e-4, 500)
plt.figure(figsize=(5.4, 3.4))
plt.plot(lam_axis, -np.log(lam_axis), color=NDSU_GREEN, lw=2.2, label="$y = 1$:  $-\\log \\lambda$")
plt.plot(lam_axis, -np.log(1 - lam_axis), color=ACCENT, lw=2.2, label="$y = 0$:  $-\\log(1 - \\lambda)$")
plt.axhline(np.log(2), color="gray", ls=":", lw=1)
plt.text(0.52, np.log(2) + 0.15, r"$\log 2$ (always guess $\lambda = 0.5$)", fontsize=8, color="gray")
plt.ylim(0, 5); plt.xlabel(r"$\lambda = \mathrm{sig}[\mathrm{f}[x, \phi]]$"); plt.ylabel("loss")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Example 3: Multiclass Classification


`fig-multiclass`


In [ ]:
TARGETS = {1: [-2.0,  3.0, -1.0, -3.0, -4.0],
           2: [-3.0, -1.5,  2.5,  0.5, -1.0],
           3: [-4.0, -3.0, -1.0,  3.0,  3.5]}

theta_m, _, _ = pwl_network(KNOTS, TARGETS[1])          # the hidden layer is shared
phi0_m = np.array([pwl_network(KNOTS, v)[1] for v in TARGETS.values()])
Phi_m  = np.array([pwl_network(KNOTS, v)[2] for v in TARGETS.values()])

x_m = np.linspace(0, 2, 1000)
H_m = np.maximum(theta_m[:, 0:1] + theta_m[:, 1:2] * x_m[None, :], 0)
F_m = phi0_m[:, None] + Phi_m @ H_m                      # K x N logits

def softmax(F):                                          # equation (5.22), shifted for stability
    E = np.exp(F - F.max(axis=0, keepdims=True))
    return E / E.sum(axis=0, keepdims=True)

S_m = softmax(F_m)
colors = [ACCENT, STEEL, NDSU_GREEN]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for k in range(3):
    axes[0].plot(x_m, F_m[k], color=colors[k], lw=2.1, label=rf"$\mathrm{{f}}_{k+1}[x, \phi]$")
    axes[1].plot(x_m, S_m[k], color=colors[k], lw=2.1, label=rf"$\lambda_{k+1} = Pr(y={k+1}\mid x)$")
winner = S_m.argmax(axis=0)
for k in range(3):
    axes[1].fill_between(x_m, 0, 1, where=(winner == k), color=colors[k], alpha=0.07)
axes[0].axhline(0, color="k", lw=0.6)
axes[0].set_title("a) network outputs (logits)"); axes[0].set_ylabel(r"$\mathrm{f}_k[x, \phi]$")
axes[1].set_title("b) after the softmax"); axes[1].set_ylabel(r"$Pr(y = k \mid x)$"); axes[1].set_ylim(0, 1)
for ax in axes:
    ax.set_xlabel("$x$"); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"every column of the softmax sums to one: {np.allclose(S_m.sum(axis=0), 1.0)}")
print(f"classes that are ever the argmax       : {sorted(int(k) + 1 for k in set(winner))}")

`softmax-properties`


In [ ]:
z = np.array([1.2, -0.4, 2.0])
print("softmax is invariant to adding a constant to every logit:")
for c in (0.0, 5.0, -100.0):
    print(f"   z + {c:>7}: {softmax((z + c)[:, None]).ravel().round(6)}")
print("\n=> the K logits carry only K - 1 degrees of freedom; the model can never")
print("   identify their absolute level, only their differences.\n")

big = np.array([1000.0, 1001.0, 999.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(big) / np.exp(big).sum()
print("the exponentials overflow if you implement equation (5.22) literally:")
print(f"   exp[z] / sum exp[z] with z = {big}  ->  {naive}")
print(f"   subtracting max[z] first                       ->  {softmax(big[:, None]).ravel().round(6)}")
print(f"   log Pr(y = 2) by log-sum-exp, equation (5.24)  ->  {big[1] - (big.max() + np.log(np.exp(big - big.max()).sum())):.6f}"
      f"   (= log {softmax(big[:, None]).ravel()[1]:.6f})")

## Cross-Entropy Loss


`cross-entropy-kl`


In [ ]:
labels = np.random.choice(4, size=500, p=[0.50, 0.25, 0.15, 0.10])
q = np.bincount(labels, minlength=4) / len(labels)      # empirical distribution, eq. (5.28)
p = np.array([0.40, 0.30, 0.20, 0.10])                  # a model distribution

H_cross = -np.sum(q * np.log(p))                        # cross-entropy H[q, p]
H_q     = -np.sum(q * np.log(q))                        # entropy H[q]
nll     = -np.sum(np.log(p[labels]))                    # equation (5.31)

print(f"empirical q               : {q.round(3)}")
print(f"model p                   : {p.round(3)}\n")
print(f"cross-entropy H[q, p]     : {H_cross:.6f}")
print(f"negative log-likelihood/I : {nll / len(labels):.6f}   <- the same number")
print(f"\nentropy H[q]              : {H_q:.6f}")
print(f"KL[q || p] = H[q,p] - H[q]: {H_cross - H_q:.6f}   >= 0, zero only when p = q")
print(f"\nH[q, q], i.e. the best any model could do on this sample: {H_q:.6f}")

## Losses in PyTorch


`pytorch-losses`


In [ ]:
torch.manual_seed(774)
I, K = 8, 5

f1, y1 = torch.randn(I, 1), torch.randn(I, 1)
manual_mse = ((y1 - f1) ** 2).sum()

f2 = torch.randn(I)
y2 = (torch.rand(I) < 0.5).float()
s = torch.sigmoid(f2)
manual_bce = (-(1 - y2) * torch.log(1 - s) - y2 * torch.log(s)).sum()

f3 = torch.randn(I, K)
y3 = torch.randint(0, K, (I,))
manual_ce = -(f3[range(I), y3] - torch.logsumexp(f3, dim=1)).sum()

print(f"{'loss':<24}{'PyTorch':>12}{'our equation':>15}{'equation':>11}")
for name, torch_val, ours, eq in [
    ("MSELoss",            nn.MSELoss(reduction="sum")(f1, y1),           manual_mse, "(5.11)"),
    ("BCEWithLogitsLoss",  nn.BCEWithLogitsLoss(reduction="sum")(f2, y2), manual_bce, "(5.20)"),
    ("CrossEntropyLoss",   nn.CrossEntropyLoss(reduction="sum")(f3, y3),  manual_ce,  "(5.24)"),
]:
    print(f"{name:<24}{torch_val.item():>12.6f}{ours.item():>15.6f}{eq:>11}")

`logit-trap`


In [ ]:
print(f"{'logit z':>9}{'BCELoss(sigmoid(z))':>22}{'BCEWithLogitsLoss(z)':>23}{'exact -log sig[z]':>20}"
      f"{'dL/dz, BCELoss':>17}{'dL/dz, logits':>15}")
for z_val in (-5.0, -20.0, -80.0, -200.0, -500.0):
    t_a = torch.tensor([z_val], requires_grad=True)
    t_b = torch.tensor([z_val], requires_grad=True)
    one = torch.tensor([1.0])                       # the true label is 1, so the loss is -log sig[z]
    L_a = nn.BCELoss()(torch.sigmoid(t_a), one)
    L_b = nn.BCEWithLogitsLoss()(t_b, one)
    g_a = torch.autograd.grad(L_a, t_a)[0].item()
    g_b = torch.autograd.grad(L_b, t_b)[0].item()
    print(f"{z_val:>9.0f}{L_a.item():>22.4f}{L_b.item():>23.4f}{-z_val:>20.4f}{g_a:>17.4f}{g_b:>15.4f}")